## LOAD SAPS Shapefile

In [5]:
import pandas as pd
import geopandas as gpd

In [34]:
station_data = gpd.read_file("../data/Police_bounds.shp")

station_data.head()



,COMPNT_NM,CREATE_DT,VERSION,geometry
0,BOTSHABELO,20251205,1.4.2,"POLYGON ((26.77137 -29.21403, 26.7733 -29.2217..."
1,KHUBUSIDRIFT,20251205,1.4.2,"POLYGON ((27.72842 -32.53051, 27.72842 -32.531..."
2,STUTTERHEIM,20251205,1.4.2,"POLYGON ((27.50201 -32.44217, 27.49884 -32.465..."
3,MOTHERWELL,20251205,1.4.2,"POLYGON ((25.61061 -33.81772, 25.60713 -33.822..."
4,KWADWESI,20251205,1.4.2,"POLYGON ((25.45586 -33.83107, 25.4666 -33.8334..."


## Load crime stats excel

In [35]:
crime_data = pd.read_excel("../data/2025-2026_-_3rd_Quarter_WEB.xlsx", sheet_name="RAW Data", header=2)
crime_data.columns = (
    crime_data.columns
    .str.strip()
    .str.replace("\n", " ")
    .str.replace(" ", "_")
)
crime_data.head()


,Crime_Category_National_contribution_placement,Crime_Category_Provincial_contribution_placement,Comp_level,Station_Crime_Category,Station,District,Province,Crime_Category,Code,NaN,...,NaN,NaN,October_2025_to__December_2025,National_contribution_placement,National_count_diff_placement,Provincial_contribution_placement,Provincial_count_diff_placement,Count_direction,Count_offence_group,No
0,Murder Station 595,Eastern Cape Murder Station 109,Station,Aberdeen Murder,Aberdeen,Sarah Baartman District,Eastern Cape,Murder,1,1.0,...,2.0,0.0,2.0,595.0,448.0,109.0,83.0,Stabilized,17 Community reported serious Crime,7085.0
1,Attempted murder Station 864,Eastern Cape Attempted murder Station 128,Station,Aberdeen Attempted murder,Aberdeen,Sarah Baartman District,Eastern Cape,Attempted murder,2,0.0,...,0.0,0.0,0.0,864.0,555.0,128.0,84.0,Stabilized,17 Community reported serious crime,7086.0
2,Culpable homicide Station 830,Eastern Cape Culpable homicide Station 120,Station,Aberdeen Culpable homicide,Aberdeen,Sarah Baartman District,Eastern Cape,Culpable homicide,3,1.0,...,0.0,0.0,1.0,830.0,1141.0,120.0,197.0,Decreased,0,7087.0
3,Robbery with aggravating circumstances Station...,Eastern Cape Robbery with aggravating circumst...,Station,Aberdeen Robbery with aggravating circumstances,Aberdeen,Sarah Baartman District,Eastern Cape,Robbery with aggravating circumstances,4,0.0,...,0.0,0.0,0.0,1141.0,721.0,191.0,135.0,Decreased,17 Community reported serious crime,7088.0
4,Common robbery Station 664,Eastern Cape Common robbery Station 77,Station,Aberdeen Common robbery,Aberdeen,Sarah Baartman District,Eastern Cape,Common robbery,6,0.0,...,0.0,1.0,2.0,664.0,416.0,77.0,67.0,Stabilized,17 Community reported serious crime,7089.0


In [ ]:
crime_data["Station"] = crime_data["Station"].astype(str)
crime_data = crime_data[crime_data["Station"].str.contains("[A-Za-z]", na=False)]
crime_data["Station"].unique()
crime_data.head()

count_col = [c for c in crime_data.columns if 'October_2025' in str(c)][0]
crime_data_clean = crime_data[['Station', 'District', 'Crime_Category', count_col, 'National_contribution_placement', 'Provincial_contribution_placement', 'Count_direction']].copy()
crime_data_clean.columns = ['Station', 'District', 'Crime_Type', 'Crime_Count', 'National_placement', 'Provincial_placement', 'Count_direction']
crime_data_clean['Station'] = crime_data_clean['Station'].str.strip().str.upper()
crime_data_clean['Crime_Type'].unique()
# grouped_crime_data = crime_data_clean.groupby(['Station', 'Crime_Type']).agg({
#     'Crime_Count': ['sum', 'mean', 'count']
# }).reset_index()
# crime_data_clean.head(50)
X = crime_data_clean.groupby('Crime_Type')['Crime_Count'].sum().sort_values(ascending=False)
X.head()

Crime_Type
17 Community reported serious Crime            1543744.0
Contact crime (Crimes against the person)       700840.0
Other serious Crime                             422832.0
Crime detected as a result of police action     361512.0
Property-related Crime                          302364.0
Name: Crime_Count, dtype: float64

In [37]:
unique_crimes = sorted(crime_data_clean['Crime_Type'].dropna().unique())

for crime in unique_crimes:
    print(crime)
    

17 Community reported serious Crime
Abduction
All theft not mentioned elsewhere
Arson
Assault with the intent to inflict grievous bodily harm
Attempted murder
Attempted sexual offences
Bank robbery
Burglary at non-residential premises
Burglary at residential premises
Carjacking
Commercial crime
Common assault
Common robbery
Contact crime (Crimes against the person)
Contact sexual offences
Contact-related Crime
Crime detected as a result of police action
Crimen injuria
Culpable homicide
Driving under the influence of alcohol or drugs
Drug-related crime
Illegal possession of firearms and ammunition
Kidnapping
Malicious damage to property
Murder
Neglect and ill-treatment of children
Other serious Crime
Property-related Crime
Public violence
Rape
Robbery at non-residential premises
Robbery at residential premises
Robbery of cash in transit
Robbery with aggravating circumstances
Sexual assault
Sexual offences
Sexual offences detected as a result of police action
Shoplifting
Stock-theft
TRIO

In [38]:
categories_to_remove = [
    '17 Community reported serious Crime',
    'Contact crime (Crimes against the person)',
    'Contact-related Crime',
    'Property-related Crime',
    'Other serious Crime',
    'TRIO Crime',
    'Crime detected as a result of police action',
    'Sexual offences',
    'Drug-related crime'
]

In [39]:
crime_data_clean = crime_data_clean[
    ~crime_data_clean['Crime_Type'].isin(categories_to_remove)
]
crime_data_clean.head()
crime_data_clean = crime_data_clean.dropna(subset=['Crime_Type'])
print(crime_data_clean['Crime_Type'].unique())

<StringArray>
[                                                 'Murder',
                                        'Attempted murder',
                                       'Culpable homicide',
                  'Robbery with aggravating circumstances',
                                          'Common robbery',
                                         'Public violence',
                                                    'Rape',
                                          'Sexual assault',
                                          'Crimen injuria',
                   'Neglect and ill-treatment of children',
                                              'Kidnapping',
                                               'Abduction',
 'Assault with the intent to inflict grievous bodily harm',
                                          'Common assault',
                    'Burglary at non-residential premises',
                        'Burglary at residential premises',
                          

In [2]:
from geopy.geocoders import Nominatim



geolocator = Nominatim(user_agent="kaggle_learn", timeout=10)
location = geolocator.geocode("Kayamandi Secondary School")

print(location.point)
print(location.address)
point = location.point
print("Latitude:", point.latitude)
print("Longitude:", point.longitude)

    

33 55m 24.0208s S, 18 50m 54.3642s E
Kayamandi Secondary School, School Crescent, Stellenbosch Ward 15, Stellenbosch, Stellenbosch Local Municipality, Cape Winelands District Municipality, Western Cape, 7599, South Africa
Latitude: -33.9233391
Longitude: 18.8484345


In [ ]:
import requests, certifi
from bs4 import BeautifulSoup
import urllib3
import os

url = "https://www.saps.gov.za/services/crimestats.php"
DOWNLOAD_DIR = r"C:\Users\DELL\Downloads"

# Disable ssl verification
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

response = requests.get(url, verify=False)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')





table = soup.find('table', class_='table')
if not table:
    raise ValueError("Table not found on page")

rows = table.find_all('tr')

data_row = rows[1] if rows else []
cols = data_row.find_all('td')
spreadsheet_download_tag = cols[-1].find("a")
spreadsheet_download_link = spreadsheet_download_tag.get('href') if spreadsheet_download_tag else None

if not spreadsheet_download_link:
    raise ValueError("No link to download")
file_url = "https://www.saps.gov.za/services/" + spreadsheet_download_link
filename = os.path.basename(spreadsheet_download_link)
file_path = os.path.join(DOWNLOAD_DIR, filename)



In [16]:
response = requests.get(file_url, verify=False)

with open(file_path, "wb") as f:
    f.write(response.content)

print(f"Saved to: {file_path}")

Saved to: C:\Users\DELL\Downloads\2025-2026_-_3rd_Quarter_WEB.xlsx
